# EVSL Dispatch Analytics – Exploratory Data Analysis (EDA)

**Goal:** Explore dispatch records from a steel-manufacturing context using Python, Pandas, NumPy and Matplotlib.

**Important data notes**
- `Production Date` is the source field name. Confirm its exact operational meaning before calling it dispatch date.
- Customer names are pseudonymised. Other commercial fields, including sales order references and prices, may still be sensitive.
- The source contains dispatch-line data, not order commitments, actual delivery dates, inventory snapshots or production plans. This notebook does not claim OTIF, inventory turnover or production plan adherence.
- `Total NSR (source field)` is excluded from additive totals because it may repeat an order-level amount across lines. The calculated line value below is `NSR (Rs/t) × Qty/Weight (t)`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Load the dataset
Place `dispatch_cleaned.csv` in the same folder as this notebook, or update the path below.

In [ ]:
df = pd.read_csv("dispatch_cleaned.csv")
df.head()


## 2. Understand the dataset

In [ ]:
print("Rows, columns:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types and non-null counts:")
df.info()


In [ ]:
df.describe(include="all").T


## 3. Data quality checks

In [ ]:
quality_profile = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "distinct_values": df.nunique(dropna=True)
})
quality_profile.sort_values("missing_values", ascending=False)


In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate sales-order references are not automatically errors; one order may have multiple dispatch lines.")


## 4. Prepare analytical fields

In [ ]:
df["Production Date"] = pd.to_datetime(df["Production Date"], errors="coerce")
numeric_cols = ["Qty/Weight (t)", "Pieces", "UPR (Rs/t)", "NSR (Rs/t)",
                "Freight (Rs/t)", "Discount (Rs/t)", "Dispatch Value (Rs)",
                "Thickness", "Width"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Month Period"] = df["Production Date"].dt.to_period("M").astype(str)
df["Calculated Dispatch Value (Rs)"] = df["NSR (Rs/t)"] * df["Qty/Weight (t)"]
df[["Production Date", "Customer Name", "FG Product", "Qty/Weight (t)",
    "NSR (Rs/t)", "Calculated Dispatch Value (Rs)"]].head()


## 5. Executive KPI summary

In [ ]:
total_tonnes = df["Qty/Weight (t)"].sum()
weighted_nsr = (df["NSR (Rs/t)"] * df["Qty/Weight (t)"]).sum() / df["Qty/Weight (t)"].sum()
kpis = pd.Series({
    "Dispatch records": len(df),
    "Total dispatch quantity (tonnes)": total_tonnes,
    "Average quantity per record (tonnes)": df["Qty/Weight (t)"].mean(),
    "Median quantity per record (tonnes)": df["Qty/Weight (t)"].median(),
    "Weighted average NSR (Rs/tonne)": weighted_nsr,
    "Calculated dispatch value (Rs)": df["Calculated Dispatch Value (Rs)"].sum(),
    "Unique customers": df["Customer Name"].nunique(),
    "Unique products": df["FG Product"].nunique(),
    "Unique destinations": df["Destination"].nunique()
})
kpis


## 6. Monthly dispatch trend

In [ ]:
monthly_dispatch = df.groupby("Month Period")["Qty/Weight (t)"].sum().sort_index()
ax = monthly_dispatch.plot(kind="line", marker="o", figsize=(10, 5))
ax.set_title("Monthly Dispatch Quantity Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Dispatch quantity (tonnes)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Customer analysis – Top 10 and Pareto

In [ ]:
customer_dispatch = df.groupby("Customer Name")["Qty/Weight (t)"].sum().sort_values(ascending=False)
top_customers = customer_dispatch.head(10)
ax = top_customers.sort_values().plot(kind="barh", figsize=(9, 5))
ax.set_title("Top 10 Customers by Dispatch Quantity")
ax.set_xlabel("Tonnes")
ax.set_ylabel("Customer")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
pareto = customer_dispatch.rename("tonnes").reset_index()
pareto["contribution_pct"] = pareto["tonnes"] / pareto["tonnes"].sum() * 100
pareto["cumulative_pct"] = pareto["contribution_pct"].cumsum()
fig, ax1 = plt.subplots(figsize=(11, 5))
x = np.arange(1, len(pareto) + 1)
ax1.bar(x, pareto["contribution_pct"])
ax1.set_xlabel("Customers ranked by dispatch quantity")
ax1.set_ylabel("Individual contribution (%)")
ax1.set_title("Customer Dispatch Concentration – Pareto Analysis")
ax2 = ax1.twinx()
ax2.plot(x, pareto["cumulative_pct"], marker=".")
ax2.axhline(80, linestyle="--")
ax2.set_ylabel("Cumulative contribution (%)")
ax2.set_ylim(0, 105)
plt.tight_layout()
plt.show()

customers_to_80 = int((pareto["cumulative_pct"] < 80).sum() + 1)
print(f"Customers required to reach or cross 80% cumulative dispatch: {customers_to_80}")


## 8. Product and destination analysis

In [ ]:
product_dispatch = df.groupby("FG Product")["Qty/Weight (t)"].sum().sort_values(ascending=False)
ax = product_dispatch.head(15).sort_values().plot(kind="barh", figsize=(10, 6))
ax.set_title("Top 15 Products by Dispatch Quantity")
ax.set_xlabel("Tonnes")
ax.set_ylabel("Finished-goods product")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
destination_dispatch = df.groupby("Destination")["Qty/Weight (t)"].sum().sort_values(ascending=False).head(10)
ax = destination_dispatch.sort_values().plot(kind="barh", figsize=(9, 5))
ax.set_title("Top 10 Destinations by Dispatch Quantity")
ax.set_xlabel("Tonnes")
ax.set_ylabel("Destination")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 9. Quality, shift and interval analysis

In [ ]:
for col, title in [("Quality", "Dispatch Quantity by Quality"),
                   ("Shift", "Dispatch Quantity by Shift"),
                   ("Interval", "Dispatch Quantity by Interval")]:
    summary = df.groupby(col)["Qty/Weight (t)"].sum().sort_values(ascending=False)
    ax = summary.sort_values().plot(kind="barh", figsize=(9, max(4, 0.5 * len(summary))))
    ax.set_title(title)
    ax.set_xlabel("Tonnes")
    ax.set_ylabel(col)
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


## 10. Distribution and outlier exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Qty/Weight (t)"].dropna(), bins=30)
axes[0].set_title("Dispatch Line Quantity Distribution")
axes[0].set_xlabel("Tonnes")
axes[0].set_ylabel("Frequency")
axes[1].hist(df["NSR (Rs/t)"].dropna(), bins=30)
axes[1].set_title("NSR per Tonne Distribution")
axes[1].set_xlabel("Rs/tonne")
axes[1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
q1 = df["Qty/Weight (t)"].quantile(0.25)
q3 = df["Qty/Weight (t)"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
quantity_outliers = df[(df["Qty/Weight (t)"] < lower) | (df["Qty/Weight (t)"] > upper)]
print("IQR lower bound:", lower)
print("IQR upper bound:", upper)
print("Potential quantity outlier rows:", len(quantity_outliers))
quantity_outliers[["Production Date", "Customer Name", "FG Product", "Qty/Weight (t)"]].head(20)


## 11. Numeric correlation

In [ ]:
corr_cols = ["Qty/Weight (t)", "Pieces", "UPR (Rs/t)", "NSR (Rs/t)",
             "Freight (Rs/t)", "Discount (Rs/t)", "Thickness", "Width"]
corr = df[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(corr, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlation Matrix")
fig.colorbar(image, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()


## 12. Save analysis outputs

In [ ]:
monthly_dispatch.to_csv("monthly_dispatch_summary.csv")
customer_dispatch.to_csv("customer_dispatch_summary.csv")
product_dispatch.to_csv("product_dispatch_summary.csv")
destination_dispatch.to_csv("destination_dispatch_summary.csv")
quality_profile.to_csv("data_quality_profile.csv")
print("Summary CSV files saved.")


## 13. Business interpretation – complete after reviewing your own outputs

Write evidence-based findings here:
1. Which month has the highest dispatch quantity?
2. Which customer contributes the most, and how concentrated is dispatch among the top customers?
3. Which product and destination lead dispatch volume?
4. How is dispatch distributed across quality categories and shifts?
5. Are there unusually large or small dispatch lines worth investigating?
6. Which numeric relationships appear strongest? Remember: correlation does not prove causation.

**Do not infer** on-time delivery, OTIF, inventory turnover, stock-outs, or production plan adherence from this dispatch-only file. Those require additional source data.
